In [69]:
import re
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
import matplotlib.pyplot as plt

## **Next Word Prediction Using Recurrent Neural Network (RNN)**

### Q1 Load And Explore and Clean The Dataset.

In [70]:
# Check files in the current working directory
for file in Path.cwd().iterdir():
    print(file.name)

print("===================================================================================================================")
# Load the text corpus
corpus_path = Path("corpus.txt")
text = corpus_path.read_text(encoding="utf-8")

# Display the first 500 characters
print(text[:500])

corpus.txt
rnn_model.ipynb
Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, “and what is the use of a book,” thought Alice “without pictures or conversations?”

So she was considering in her own mind (as well as she could, for the hot day made her feel very sleepy and stupid), whether the pleasure of making a daisy-chain would be worth the trouble of gett


- We Have Loaded The Corpus with the help of pathlib and encoding with the utf-8.

In [71]:
# Exploring The Dataset
print("Total Characters:", len(text))
print("Total Words:", len(text.split()))
print("Total Lines:", len(text.splitlines()))

Total Characters: 11224
Total Words: 2141
Total Lines: 47


In [72]:
# Check unique words before preprocessing
print("Unique Words:", len(set(text.split())))

Unique Words: 799


- This helps us identify capitalization, punctuation, line breaks, and other text patterns that may need cleaning.

In [73]:
# Convert text to lowercase
cleaned_text = text.lower()

# Remove unwanted characters while keeping basic punctuation
cleaned_text = re.sub(r"[^a-z0-9\s.,!?;:'\"-]", "", cleaned_text)

# Normalize extra whitespace
cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

# Remove spaces from the beginning and end
text = text.strip()
print(cleaned_text)

alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, and what is the use of a book, thought alice without pictures or conversations? so she was considering in her own mind as well as she could, for the hot day made her feel very sleepy and stupid, whether the pleasure of making a daisy-chain would be worth the trouble of getting up and picking the daisies, when suddenly a white rabbit with pink eyes ran close by her. there was nothing so very remarkable in that; nor did alice think it so very much out of the way to hear the rabbit say to itself, oh dear! oh dear! i shall be late! when she thought it over afterwards, it occurred to her that she ought to have wondered at this, but at the time it all seemed quite natural; but when the rabbit actually took a watch out of its waistcoat-pocket, and looked at it, and then hurried

- lower() makes words consistent, so Alice and alice become the same token.
- re.sub() removes unwanted characters while retaining common punctuation.
- Extra spaces and line breaks are normalized.

In [74]:
print("Total Characters:", len(cleaned_text))
print("Total Words:", len(cleaned_text.split()))
print("Total Lines:", len(cleaned_text.splitlines()))

Total Characters: 11066
Total Words: 2141
Total Lines: 1


### Q2 How do we tokenize the text and How do we create input sequences and target words?

In [75]:
# Split the cleaned corpus at a word boundary
split_index = int(len(cleaned_text) * 0.80)    # The first 80% of the text is used for training, and the final 20% is reserved for testing.
split_index = cleaned_text.rfind(" ", 0, split_index)

train_text = cleaned_text[:split_index]
test_text = cleaned_text[split_index:].strip()

# Initialize the tokenizer
tokenizer = tf.keras.preprocessing.text.Tokenizer(oov_token="<OOV>")

# Learn vocabulary from training text only
tokenizer.fit_on_texts([train_text])

# Convert both text splits into numerical word IDs
train_sequences = tokenizer.texts_to_sequences([train_text])[0]
test_sequences = tokenizer.texts_to_sequences([test_text])[0]

print("Vocabulary Size:", len(tokenizer.word_index))
print("Training Tokens:", len(train_sequences))
print("Testing Tokens:", len(test_sequences))

print("\nFirst 10 training token IDs:", train_sequences[:10])

Vocabulary Size: 536
Training Tokens: 1727
Testing Tokens: 433

First 10 training token IDs: [11, 7, 225, 4, 50, 16, 226, 9, 227, 51]


In [76]:
# Display the first 10 words with their IDs
print("First 10 word mappings:")

for word, index in list(tokenizer.word_index.items())[:10]:
    print(word, "->", index)

First 10 word mappings:
<OOV> -> 1
the -> 2
she -> 3
to -> 4
and -> 5
it -> 6
was -> 7
a -> 8
of -> 9
i -> 10


- Tokenizer() creates a tokenizer object.
- fit_on_texts() identifies unique words and assigns each a numerical ID.
- word_index stores the mapping between words and their IDs.
- The most frequent words generally receive lower IDs.
- The text is now represented as a sequence of integers. Each integer corresponds to a word in the vocabulary.

In [77]:
# Creating Sequences To Create Examples For RNN Model
def create_sequences(token_sequence, sequence_length=5):
    input_sequences = []

    for i in range(sequence_length, len(token_sequence)):
        sequence = token_sequence[i - sequence_length:i + 1]
        input_sequences.append(sequence)

    return input_sequences

train_input_sequences = create_sequences(train_sequences,sequence_length=5)
test_input_sequences = create_sequences(test_sequences,sequence_length=5)

print("Training sequences:", len(train_input_sequences))
print("Testing sequences:", len(test_input_sequences))
print("Sample training sequences:", train_input_sequences[:5])

Training sequences: 1722
Testing sequences: 428
Sample training sequences: [[11, 7, 225, 4, 50, 16], [7, 225, 4, 50, 16, 226], [225, 4, 50, 16, 226, 9], [4, 50, 16, 226, 9, 227], [50, 16, 226, 9, 227, 51]]


- We are creating sequences because an RNN needs examples of previous words and the next word it should predict.
- The function Is Selecting 5 previous words plus one target word to tell the RNN what sequences the corpus is holding.

### Q3 How do we separate the input words (X) and target words (y)?

In [78]:
# Convert sequences into NumPy arrays
train_array = np.array(train_input_sequences)
test_array = np.array(test_input_sequences)

# Separate input words and target words
X_train = train_array[:, :-1]
y_train = train_array[:, -1]

X_test = test_array[:, :-1]
y_test = test_array[:, -1]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (1722, 5)
y_train shape: (1722,)
X_test shape: (428, 5)
y_test shape: (428,)


- X_train contains the five preceding word IDs.
- y_train contains the next word ID the model must predict.
- X_test and y_test are used to evaluate the model on held-out text.
- [:, :-1] selects all columns except the last.
- [:, -1] selects only the last column.

In [79]:
# Verifying The Data
print("Sample input IDs:", X_train[0])
print("Target ID:", y_train[0])

print("Input words:",tokenizer.sequences_to_texts([X_train[0]])[0])
print("Target word:",tokenizer.index_word.get(y_train[0], "<UNKNOWN>"))

Sample input IDs: [ 11   7 225   4  50]
Target ID: 16
Input words: alice was beginning to get
Target word: very


- We Checked That your input consists of words and the target is the next word.

In [82]:
# Define vocabulary size
vocab_size = len(tokenizer.word_index) + 1

# Build the RNN model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(5,)),

    tf.keras.layers.Embedding(
    input_dim=vocab_size,
    output_dim=64,
    mask_zero=True),

    tf.keras.layers.SimpleRNN(128),
    tf.keras.layers.Dense(vocab_size,activation="softmax")
])
# Display model architecture
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 5, 64)          │        34,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 128)            │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 537)            │        69,273 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 128,345 (501.35 KB)

 Trainable params: 128,345 (501.35 KB)

 Non-trainable params: 0 (0.00 B)